# Entrenamiento tabular

In [1]:
from pathlib import Path
import json

import numpy as np

from src.training.train import load_train_arrays, train_tabular_model, save_trained_pipeline

In [2]:
# CONFIGURACIÓN
DATA_DIR = Path("/home/dani/github/profecia/data")

temporal_resolution = "annual"   # "monthly" o "annual"
split_type = "landcover"         # "landcover" o "climate"

MODEL_NAME = "rf"                # "rf", "hgb", "mlp"
SCALER_NAME = None               # None, "standard", "robust"

RANDOM_STATE = 42
MMAP_MODE = "r"

SPLITS_DIR = DATA_DIR / "splits" / temporal_resolution / split_type
INPUT_DIR = SPLITS_DIR

MODELS_DIR = DATA_DIR / "models" / temporal_resolution / split_type
MODEL_PREFIX = f"{temporal_resolution}_{split_type}_{MODEL_NAME}"

print("INPUT_DIR   :", INPUT_DIR)
print("MODELS_DIR  :", MODELS_DIR)
print("MODEL_PREFIX:", MODEL_PREFIX)

INPUT_DIR   : /home/dani/github/profecia/data/splits/annual/landcover
MODELS_DIR  : /home/dani/github/profecia/data/models/annual/landcover
MODEL_PREFIX: annual_landcover_rf


In [3]:
# HIPERPARÁMETROS

RF_PARAMS = {
    "n_estimators": 100,
    "max_depth": 20,
    "min_samples_leaf": 5,
    "max_features": "sqrt",
    "n_jobs": -1,
    "bootstrap": True,
    "max_samples": 0.3,
}

HGB_PARAMS = {
    "learning_rate": 0.1,
    "max_iter": 200,
    "max_depth": None,
    "min_samples_leaf": 20,
}

MLP_PARAMS = {
    "hidden_layer_sizes": (128, 64),
    "activation": "relu",
    "solver": "adam",
    "alpha": 1e-4,
    "learning_rate": "adaptive",
    "learning_rate_init": 1e-3,
    "max_iter": 300,
    "early_stopping": True,
    "validation_fraction": 0.1,
}

MODEL_PARAM_MAP = {
    "rf": RF_PARAMS,
    "hgb": HGB_PARAMS,
    "mlp": MLP_PARAMS,
}

if MODEL_NAME not in MODEL_PARAM_MAP:
    raise ValueError("MODEL_NAME debe ser 'rf', 'hgb' o 'mlp'")

MODEL_PARAMS = MODEL_PARAM_MAP[MODEL_NAME]

print("MODEL_NAME :", MODEL_NAME)
print("SCALER_NAME:", SCALER_NAME)
print("MODEL_PARAMS:")
print(json.dumps(MODEL_PARAMS, indent=2, ensure_ascii=False, default=str))

MODEL_NAME : rf
SCALER_NAME: None
MODEL_PARAMS:
{
  "n_estimators": 100,
  "max_depth": 20,
  "min_samples_leaf": 5,
  "max_features": "sqrt",
  "n_jobs": -1,
  "bootstrap": true,
  "max_samples": 0.3
}


## Cargar arrays de train

In [4]:
X_train, y_train, dataset_metadata = load_train_arrays(
    input_dir=INPUT_DIR,
    mmap_mode=MMAP_MODE,
)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

print("\nDataset metadata:")
print(json.dumps(dataset_metadata, indent=2, ensure_ascii=False))

X_train shape: (1175123, 6)
y_train shape: (1175123,)

Dataset metadata:
{
  "prefix": "landcover_annual",
  "n_train": 1175123,
  "n_test": 130572,
  "n_features": 6,
  "feature_names": [
    "SM1",
    "SM2",
    "TP",
    "T2M",
    "SSRD",
    "VPD"
  ],
  "target": "LAI",
  "target_dtype": "float32",
  "feature_dtypes": {
    "SM1": "float32",
    "SM2": "float32",
    "TP": "float32",
    "T2M": "float32",
    "SSRD": "float32",
    "VPD": "float32"
  },
  "n_time": 41,
  "time_start": "1982-01-01 00:00:00",
  "time_end": "2022-01-01 00:00:00",
  "time_values": [
    "1982-01-01 00:00:00",
    "1983-01-01 00:00:00",
    "1984-01-01 00:00:00",
    "1985-01-01 00:00:00",
    "1986-01-01 00:00:00",
    "1987-01-01 00:00:00",
    "1988-01-01 00:00:00",
    "1989-01-01 00:00:00",
    "1990-01-01 00:00:00",
    "1991-01-01 00:00:00",
    "1992-01-01 00:00:00",
    "1993-01-01 00:00:00",
    "1994-01-01 00:00:00",
    "1995-01-01 00:00:00",
    "1996-01-01 00:00:00",
    "1997-01-01 00:

## Checks rápidos

In [5]:
if X_train.ndim != 2:
    raise ValueError(f"X_train debe ser 2D. Recibido: {X_train.shape}")

if y_train.ndim != 1:
    raise ValueError(f"y_train debe ser 1D. Recibido: {y_train.shape}")

if X_train.shape[0] != y_train.shape[0]:
    raise ValueError("X_train e y_train no tienen el mismo número de filas.")

if "n_features" in dataset_metadata:
    if X_train.shape[1] != dataset_metadata["n_features"]:
        raise ValueError("El número de features no coincide con dataset_metadata.")

print("Shapes correctos.")

Shapes correctos.


In [6]:
n_nan_x = int(np.isnan(X_train).sum())
n_nan_y = int(np.isnan(y_train).sum())

print("NaN en X_train:", n_nan_x)
print("NaN en y_train:", n_nan_y)

if n_nan_x != 0:
    raise ValueError("Hay NaN en X_train.")
if n_nan_y != 0:
    raise ValueError("Hay NaN en y_train.")

print("No hay NaN.")

NaN en X_train: 0
NaN en y_train: 0
No hay NaN.


In [7]:
print("y_train min :", float(np.min(y_train)))
print("y_train max :", float(np.max(y_train)))
print("y_train mean:", float(np.mean(y_train)))
print("y_train std :", float(np.std(y_train)))

y_train min : 0.0
y_train max : 5.93981409072876
y_train mean: 1.2224721908569336
y_train std : 1.2065738439559937


## Entrenar modelo

In [8]:
train_result = train_tabular_model(
    X_train=X_train,
    y_train=y_train,
    model_name=MODEL_NAME,
    scaler_name=SCALER_NAME,
    random_state=RANDOM_STATE,
    **MODEL_PARAMS,
)

print("Modelo entrenado:", train_result["train_info"]["model_name"])
print("Modelo pedido   :", train_result["train_info"]["model_name_requested"])
print("Scaler          :", train_result["train_info"]["scaler_name"])

Modelo entrenado: RandomForestRegressor
Modelo pedido   : rf
Scaler          : none


In [9]:
train_result['train_info']

{'model_name_requested': 'rf',
 'model_name': 'RandomForestRegressor',
 'model_params': {'bootstrap': True,
  'ccp_alpha': 0.0,
  'criterion': 'squared_error',
  'max_depth': 20,
  'max_features': 'sqrt',
  'max_leaf_nodes': None,
  'max_samples': 0.3,
  'min_impurity_decrease': 0.0,
  'min_samples_leaf': 5,
  'min_samples_split': 2,
  'min_weight_fraction_leaf': 0.0,
  'monotonic_cst': None,
  'n_estimators': 100,
  'n_jobs': -1,
  'oob_score': False,
  'random_state': 42,
  'verbose': 0,
  'warm_start': False},
 'scaler_name': 'none',
 'random_state': 42,
 'n_rows_train': 1175123,
 'n_features': 6,
 'X_dtype': 'float32',
 'y_dtype': 'float32'}

## Guardar modelo

In [10]:
saved_paths = save_trained_pipeline(
    output_dir=MODELS_DIR,
    model=train_result["model"],
    scaler=train_result["scaler"],
    train_info=train_result["train_info"],
    dataset_metadata=dataset_metadata,
    prefix=MODEL_PREFIX,
)

saved_paths

{'model_path': '/home/dani/github/profecia/data/models/annual/landcover/annual_landcover_rf_model.joblib',
 'scaler_path': None,
 'train_info_path': '/home/dani/github/profecia/data/models/annual/landcover/annual_landcover_rf_train_info.json'}